# Cálculo de Métricas e Scores para Viagens Aéreas UFPB (Completo)

Este notebook calcula as métricas-chave e os scores de desempenho (ED1.1, ED1.2, ..., DF1.5) com base nos dados processados de viagens aéreas da UFPB.

## 1. Importações e Configurações

In [74]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

# --- Configurações --- 
ano = 2025 # Ano dos dados a serem analisados

# Caminho para o arquivo de entrada (resultado final do notebook anterior)
pasta_dados = f'dadosViagens/dados_viagens{ano}'
arquivo_entrada = os.path.join(pasta_dados, f'df_master_ufpb_aereo_{ano}.csv')

# Colunas Requeridas (Verifique se os nomes estão corretos no seu CSV)
COL_ID_VIAGEM = 'Identificador do processo de viagem'
COL_CPF_VIAJANTE = 'CPF viajante' # Essencial para métricas por viajante
COL_DISTANCIA = 'Distância (GCD)'
COL_EMISSOES = 'Emissões (KgCO2eq)'
COL_CATEGORIA_DIST = 'Categoria Distância'
COL_VIAGEM_URGENTE = 'Viagem Urgente'
COL_JUSTIFICATIVA_URGENCIA = 'Justificativa Urgência Viagem'
COLUNAS_CUSTO = ['Valor passagens', 'Valor diárias', 'Valor outros gastos'] 

# Valores para flags e categorias
VALOR_URGENTE_SIM = 'SIM'
VALOR_SEM_JUSTIFICATIVA = 'Sem informação'
CATEGORIA_EVITAVEL = 'Muito Curta (Evitável)'

# Nome do órgão para o arquivo de saída
# orgao = 'UFPB'
orgao = 'UFCG'
arquivo_entrada = os.path.join(pasta_dados, f'df_master_{orgao}_aereo_{ano}.csv')

## 2. Carregar Dados Processados

In [67]:
print(f"🔄 Carregando dados de: '{arquivo_entrada}'...")
try:
    df = pd.read_csv(arquivo_entrada)
    print(f"✅ Dados carregados com sucesso ({len(df)} viagens).")
    
    # Verificações iniciais
    print("\n--- Verificando Colunas Essenciais ---")
    colunas_essenciais = [COL_ID_VIAGEM, COL_CPF_VIAJANTE, COL_DISTANCIA, COL_EMISSOES, 
                         COL_CATEGORIA_DIST, COL_VIAGEM_URGENTE, COL_JUSTIFICATIVA_URGENCIA] + COLUNAS_CUSTO
    colunas_faltantes = [col for col in colunas_essenciais if col not in df.columns]
    if colunas_faltantes:
        print(f"❌ ERRO: Colunas essenciais não encontradas no arquivo CSV: {colunas_faltantes}")
        print("   Verifique os nomes das colunas no arquivo ou nas configurações do notebook.")
        # Preenche colunas faltantes com NaN/0 para evitar erros posteriores, mas os cálculos podem ser afetados
        for col in colunas_faltantes: df[col] = 0 if col in COLUNAS_CUSTO + [COL_DISTANCIA, COL_EMISSOES] else np.nan
        print("   -> Colunas faltantes preenchidas com valores padrão (NaN/0).")
        
    else:
        print("✅ Todas as colunas essenciais encontradas.")
        
    print("\n--- Amostra dos Dados --- ")
    # Mostra colunas chave
    cols_amostra = [COL_ID_VIAGEM, COL_CPF_VIAJANTE, COL_DISTANCIA, COL_EMISSOES, COL_CATEGORIA_DIST, COL_VIAGEM_URGENTE]
    cols_amostra_presentes = [col for col in cols_amostra if col in df.columns]
    print(df[cols_amostra_presentes].head())
    # df.info() # Descomente para ver mais detalhes

except FileNotFoundError:
    print(f"❌ ERRO: Arquivo '{arquivo_entrada}' não encontrado.")
    df = pd.DataFrame() # Cria DF vazio
except Exception as e:
    print(f"❌ Ocorreu um erro ao carregar os dados: {e}")
    df = pd.DataFrame() # Cria DF vazio

🔄 Carregando dados de: 'dadosViagens/dados_viagens2023\df_master_UFCG_aereo_2023.csv'...
✅ Dados carregados com sucesso (320 viagens).

--- Verificando Colunas Essenciais ---
✅ Todas as colunas essenciais encontradas.

--- Amostra dos Dados --- 
   Identificador do processo de viagem    CPF viajante  Distância (GCD)  \
0                             18548551  ***.117.904-**      3412.713516   
1                             18559255  ***.756.248-**      4230.516148   
2                             18559668  ***.776.948-**      4230.516148   
3                             18657968  ***.031.664-**      3424.980642   
4                             18666746  ***.690.984-**     11373.956459   

   Emissões (KgCO2eq) Categoria Distância Viagem Urgente  
0            624.0807     Curta Distância            NÃO  
1            846.5610     Longa Distância            NÃO  
2            846.5610     Longa Distância            NÃO  
3            626.3240     Curta Distância            NÃO  
4       

## 3. Preparação dos Dados para Cálculo

In [68]:
metrics_brutas = {} # Dicionário para guardar os resultados

if not df.empty:
    print("\n--- Preparando Dados para Cálculo ---")

    # --- Garante Tipos Numéricos para Cálculos --- 
    print("🔄 Convertendo colunas relevantes para numérico...")
    colunas_numericas = [COL_DISTANCIA, COL_EMISSOES] + [col for col in COLUNAS_CUSTO if col in df.columns]
    for col in colunas_numericas:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '.', regex=False), errors='coerce').fillna(0)
            print(f"   - Coluna '{col}' convertida.")
        else:
             print(f"   ⚠️ Coluna numérica '{col}' não encontrada, pulando conversão.")
             

    # --- Calcula Custo Total por Viagem --- 
    custo_cols_presentes = [col for col in COLUNAS_CUSTO if col in df.columns]
    if custo_cols_presentes:
        df['Custo_Total_Viagem'] = df[custo_cols_presentes].sum(axis=1)
        print("✅ Coluna 'Custo_Total_Viagem' calculada.")
    else:
        df['Custo_Total_Viagem'] = 0 # Define como 0 se não houver colunas de custo
        print("⚠️ Nenhuma coluna de custo encontrada, 'Custo_Total_Viagem' definida como 0.")
        
    # --- Identifica Viajantes Únicos --- 
    total_viajantes_unicos = 0 # Default
    if COL_CPF_VIAJANTE in df.columns:
         # Conta valores únicos não nulos na coluna CPF
         total_viajantes_unicos = df[COL_CPF_VIAJANTE].nunique() 
         print(f"✅ Encontrados {total_viajantes_unicos} viajantes únicos (baseado em '{COL_CPF_VIAJANTE}').")
    else:
         print(f"⚠️ Coluna '{COL_CPF_VIAJANTE}' não encontrada. Métricas 'por viajante' serão 0.")

    # --- Totais Gerais --- 
    total_viagens = len(df)
    total_distancia = df[COL_DISTANCIA].sum() if COL_DISTANCIA in df.columns else 0
    total_emissoes = df[COL_EMISSOES].sum() if COL_EMISSOES in df.columns else 0
    custo_total_geral = df['Custo_Total_Viagem'].sum()
    
    print("✅ Dados preparados.")

else:
    print("⚠️ DataFrame vazio, preparação pulada.")
    # Define totais como 0 se o DF estiver vazio para evitar erros
    total_viagens = 0
    total_viajantes_unicos = 0
    total_distancia = 0
    total_emissoes = 0
    custo_total_geral = 0


--- Preparando Dados para Cálculo ---
🔄 Convertendo colunas relevantes para numérico...
   - Coluna 'Distância (GCD)' convertida.
   - Coluna 'Emissões (KgCO2eq)' convertida.
   - Coluna 'Valor passagens' convertida.
   - Coluna 'Valor diárias' convertida.
   - Coluna 'Valor outros gastos' convertida.
✅ Coluna 'Custo_Total_Viagem' calculada.
✅ Encontrados 216 viajantes únicos (baseado em 'CPF viajante').
✅ Dados preparados.


## 4. Cálculo dos Indicadores (Métricas Brutas)

In [69]:
if total_viagens > 0: # Só calcula se houver dados
    print("\n--- Calculando Indicadores (Métricas Brutas) ---")
    
    # --- Dimensão 1: Emissões (ED1) ---
    print("   - Calculando ED1...")
    # ED1.1: Total Emissions (KgCO2eq)
    metrics_brutas['ED1.1_Total_Emissions_KgCO2eq'] = total_emissoes
    
    # ED1.2: Avoidable Emissions (KgCO2eq) - Emissões de viagens <= 600km
    emiss_evitaveis = 0
    if COL_CATEGORIA_DIST in df.columns and COL_EMISSOES in df.columns:
        emiss_evitaveis = df.loc[df[COL_CATEGORIA_DIST] == CATEGORIA_EVITAVEL, COL_EMISSOES].sum()
    else: print(f"   ⚠️ Não foi possível calcular ED1.2 (colunas faltantes).")
    metrics_brutas['ED1.2_Avoidable_Emissions_KgCO2eq'] = emiss_evitaveis
    
    # ED1.3: Avg Emissions per Traveler (KgCO2eq / viajante)
    avg_emiss_traveler = (total_emissoes / total_viajantes_unicos) if total_viajantes_unicos > 0 else 0
    metrics_brutas['ED1.3_Avg_Emissions_per_Traveler'] = avg_emiss_traveler
    
    # ED1.4: Avg Emissions per Trip (KgCO2eq / viagem)
    avg_emiss_trip = (total_emissoes / total_viagens) if total_viagens > 0 else 0
    metrics_brutas['ED1.4_Avg_Emissions_per_Trip'] = avg_emiss_trip

    # --- Dimensão 2: Custos (ED2) ---
    print("   - Calculando ED2...")
    # ED2.1: Total Costs (R$)
    metrics_brutas['ED2.1_Total_Costs_R$'] = custo_total_geral
    
    # ED2.2: Avg Cost per Traveler (R$ / viajante)
    avg_cost_traveler = (custo_total_geral / total_viajantes_unicos) if total_viajantes_unicos > 0 else 0
    metrics_brutas['ED2.2_Avg_Cost_per_Traveler'] = avg_cost_traveler
    
    # ED2.3: Total Trips
    metrics_brutas['ED2.3_Total_Trips'] = total_viagens
    
    # ED2.4: Avg Cost per Trip (R$ / viagem)
    avg_cost_trip = (custo_total_geral / total_viagens) if total_viagens > 0 else 0
    metrics_brutas['ED2.4_Avg_Cost_per_Trip'] = avg_cost_trip
    
    # ED2.5: Avg Cost per Km (R$ / Km)
    avg_cost_km = (custo_total_geral / total_distancia) if total_distancia > 0 else 0
    metrics_brutas['ED2.5_Avg_Cost_per_Km'] = avg_cost_km

    # --- Dimensão 3: Distância (ED3) ---
    print("   - Calculando ED3...")
    # ED3.1: Total Distance (Km)
    metrics_brutas['ED3.1_Total_Distance_Km'] = total_distancia
    
    # ED3.2: Avg Distance per Traveler (Km / viajante)
    avg_dist_traveler = (total_distancia / total_viajantes_unicos) if total_viajantes_unicos > 0 else 0
    metrics_brutas['ED3.2_Avg_Distance_per_Traveler'] = avg_dist_traveler
    
    # ED3.3: Avg Distance per Trip (Km / viagem)
    avg_dist_trip = (total_distancia / total_viagens) if total_viagens > 0 else 0
    metrics_brutas['ED3.3_Avg_Distance_per_Trip'] = avg_dist_trip

    # --- Dimensão 4: Governança (DF1) ---
    print("   - Calculando DF1...")
    # DF1.4: Urgent Trips (%)
    total_urgentes = 0
    if COL_VIAGEM_URGENTE in df.columns:
        total_urgentes = (df[COL_VIAGEM_URGENTE].astype(str).str.strip().str.upper() == VALOR_URGENTE_SIM).sum()
    else: print(f"   ⚠️ Aviso: Coluna '{COL_VIAGEM_URGENTE}' não encontrada.")
    perc_urgentes = (total_urgentes / total_viagens * 100) if total_viagens > 0 else 0
    metrics_brutas['DF1.4_Urgent_Trips_Percent'] = perc_urgentes
    metrics_brutas['DF1.4_Urgent_Trips_Count'] = total_urgentes 
        
    # DF1.5: Urgent Trips w/o Justification (%)
    urgentes_sem_justificativa_count = 0
    if COL_VIAGEM_URGENTE in df.columns and COL_JUSTIFICATIVA_URGENCIA in df.columns:
        urgente_mask = df[COL_VIAGEM_URGENTE].astype(str).str.strip().str.upper() == VALOR_URGENTE_SIM
        sem_justif_mask = df[COL_JUSTIFICATIVA_URGENCIA].astype(str).str.strip().fillna(VALOR_SEM_JUSTIFICATIVA).str.upper() == VALOR_SEM_JUSTIFICATIVA
        urgentes_sem_justificativa_count = df[urgente_mask & sem_justif_mask].shape[0]
    else: print(f"   ⚠️ Aviso: Colunas '{COL_VIAGEM_URGENTE}' ou '{COL_JUSTIFICATIVA_URGENCIA}' não encontradas.")
    perc_urgentes_sem_justif = (urgentes_sem_justificativa_count / total_viagens * 100) if total_viagens > 0 else 0
    metrics_brutas['DF1.5_Urgent_Trips_wo_Justif_Percent'] = perc_urgentes_sem_justif
    metrics_brutas['DF1.5_Urgent_Trips_wo_Justif_Count'] = urgentes_sem_justificativa_count

    print("✅ Todos os indicadores calculados.")

    # --- Cria e Exibe DataFrame de Métricas --- 
    metrics_df = pd.DataFrame(metrics_brutas.items(), columns=['Indicador', 'Valor Bruto'])
    # Função de formatação (igual à anterior)
    def format_value(row): 
        indicador = row['Indicador']; valor = row['Valor Bruto']
        if pd.isna(valor): return 'N/A'
        if 'Percent' in indicador: return f"{valor:.2f}%"
        if 'R$' in indicador or 'Cost' in indicador: return f"R$ {valor:,.2f}"
        if 'KgCO2eq' in indicador or 'Emissions' in indicador: return f"{valor:,.2f} KgCO2eq"
        if 'Km' in indicador or 'Distance' in indicador: return f"{valor:,.2f} Km"
        if 'Count' in indicador or 'Trips' in indicador: return f"{int(valor)}" 
        return f"{valor:.2f}" 
    metrics_df['Valor Formatado'] = metrics_df.apply(format_value, axis=1)
    
    print("\n--- Tabela de Métricas Brutas ---")
    with pd.option_context('display.max_rows', None): # Mostra todas as linhas
        print(metrics_df[['Indicador', 'Valor Formatado']].to_string(index=False))

else:
    print("⚠️ DataFrame vazio, cálculo de métricas pulado.")
    metrics_df = pd.DataFrame(columns=['Indicador', 'Valor Bruto'])


--- Calculando Indicadores (Métricas Brutas) ---
   - Calculando ED1...
   - Calculando ED2...
   - Calculando ED3...
   - Calculando DF1...
✅ Todos os indicadores calculados.

--- Tabela de Métricas Brutas ---
                           Indicador    Valor Formatado
       ED1.1_Total_Emissions_KgCO2eq 270,626.40 KgCO2eq
   ED1.2_Avoidable_Emissions_KgCO2eq     224.41 KgCO2eq
    ED1.3_Avg_Emissions_per_Traveler   1,252.90 KgCO2eq
        ED1.4_Avg_Emissions_per_Trip     845.71 KgCO2eq
                ED2.1_Total_Costs_R$    R$ 1,551,094.54
         ED2.2_Avg_Cost_per_Traveler        R$ 7,180.99
                   ED2.3_Total_Trips                320
             ED2.4_Avg_Cost_per_Trip        R$ 4,847.17
               ED2.5_Avg_Cost_per_Km            R$ 1.11
             ED3.1_Total_Distance_Km    1,395,655.58 Km
     ED3.2_Avg_Distance_per_Traveler        6,461.37 Km
         ED3.3_Avg_Distance_per_Trip        4,361.42 Km
          DF1.4_Urgent_Trips_Percent             49.38%
    

## 5. Definição das Constantes de Baseline

In [70]:
# # --- CONSTANTES DO BASELINE --- 
# # Substitua pelos valores corretos definidos para sua análise

# if ano == 2023:
#     BASELINE_ED1_1 = 285440.323 
#     BASELINE_ED2_1 = 1610128.775 
# elif ano == 2024:
#     BASELINE_ED1_1 = 300000.000 # Novo exemplo para 2024 (ajuste!)
#     BASELINE_ED2_1 = 1700000.000 # Novo exemplo para 2024 (ajuste!)
# elif ano == 2025:
#     BASELINE_ED1_1 = 320000.000 # Novo exemplo para 2025 (ajuste!)
#     BASELINE_ED2_1 = 1800000.000 # Novo exemplo para 2025 (ajuste!)

# print("\n--- Constantes de Baseline Definidas ---")
# print(f"   - Baseline ED1.1 (Emissões): {BASELINE_ED1_1:,.2f} KgCO2eq")
# print(f"   - Baseline ED2.1 (Custo): R$ {BASELINE_ED2_1:,.2f}")

In [71]:
# --- ETAPA 5 (REVISADA): DEFINIÇÃO DAS CONSTANTES DE BASELINE (DINÂMICO) ---
import pandas as pd
import numpy as np
import os
import glob
import re # Para limpeza

print("\n--- Definindo Constantes de Baseline ---")

# --- Valores Históricos Fixos (Fallback para o primeiro ano) ---
BASELINE_HISTORICO_ED1_1 = 285440.323 
BASELINE_HISTORICO_ED2_1 = 1738431.16 

# --- Tenta Carregar Baseline do Ano Anterior ---
ano_anterior = ano - 1
baseline_ed1_1_final = BASELINE_HISTORICO_ED1_1 # Inicializa com fallback
baseline_ed2_1_final = BASELINE_HISTORICO_ED2_1 # Inicializa com fallback
arquivo_ano_anterior_encontrado = False

# Define pasta e padrão do arquivo do ano anterior
pasta_dados_anterior = f'dadosViagens/dados_viagens{ano_anterior}'
padrao_arquivo_anterior = os.path.join(pasta_dados_anterior, f"relatorio_metricas_scores_{orgao}_{ano_anterior}_*.csv")

# Procura pelo arquivo mais recente do ano anterior
lista_arquivos_anterior = glob.glob(padrao_arquivo_anterior)
if lista_arquivos_anterior:
    arquivo_relatorio_anterior = max(lista_arquivos_anterior, key=os.path.getctime)
    print(f"🔄 Encontrado relatório do ano anterior: '{os.path.basename(arquivo_relatorio_anterior)}'. Tentando carregar...")
    try:
        df_anterior = pd.read_csv(arquivo_relatorio_anterior, sep=';', decimal=',')
        
        # Função de limpeza (igual à da Célula 8)
        def clean_numeric_value(value_in):
            if pd.isna(value_in): return np.nan
            if isinstance(value_in, (int, float)): return float(value_in)
            try:
                s = str(value_in).strip()
                s = re.sub(r'(R\$|\s?KgCO2eq|\s?Km|%)', '', s, flags=re.IGNORECASE).strip()
                is_percent = '%' in s
                s = s.replace('%', '').strip()
                if '.' in s and ',' in s and s.rfind('.') < s.rfind(','): s = s.replace('.', '').replace(',', '.')
                elif ',' in s: s = s.replace(',', '')
                num = float(s)
                return num
            except: return np.nan

        # Extrai valores de Emissões e Custo Total do ano anterior
        valor_emissao_anterior_str = df_anterior.loc[(df_anterior['Indicador/Métrica'] == 'ED1.1_Total_Emissions_KgCO2eq') & (df_anterior['Tipo'] == 'Métrica Bruta'), 'Valor'].iloc[0]
        valor_custo_anterior_str = df_anterior.loc[(df_anterior['Indicador/Métrica'] == 'ED2.1_Total_Costs_R$') & (df_anterior['Tipo'] == 'Métrica Bruta'), 'Valor'].iloc[0]
        
        valor_emissao_anterior_num = clean_numeric_value(valor_emissao_anterior_str)
        valor_custo_anterior_num = clean_numeric_value(valor_custo_anterior_str)

        # Verifica se conseguiu extrair os números
        if pd.notna(valor_emissao_anterior_num) and pd.notna(valor_custo_anterior_num):
            baseline_ed1_1_final = valor_emissao_anterior_num
            baseline_ed2_1_final = valor_custo_anterior_num
            arquivo_ano_anterior_encontrado = True
            print("✅ Baseline definido com os totais do ano anterior.")
        else:
            print("   ⚠️ Não foi possível extrair valores numéricos do arquivo do ano anterior. Usando baseline histórico.")
            
    except FileNotFoundError:
        # Isso não deveria acontecer se glob encontrou o arquivo, mas por segurança
        print(f"   ⚠️ Erro ao carregar arquivo do ano anterior (não encontrado após glob). Usando baseline histórico.")
    except (IndexError, KeyError) as e_extract:
         print(f"   ⚠️ Erro ao extrair métricas do arquivo do ano anterior: {e_extract}. Verifique as colunas/linhas do arquivo. Usando baseline histórico.")
    except Exception as e_load_prev:
        print(f"   ❌ Erro inesperado ao carregar/processar arquivo do ano anterior: {e_load_prev}. Usando baseline histórico.")
        
else: # Se não encontrou arquivo do ano anterior
    print(f"   Arquivo de relatório para {ano_anterior} não encontrado (padrão: {padrao_arquivo_anterior}). Usando baseline histórico fixo.")

# --- Define as Constantes Finais ---
BASELINE_ED1_1 = baseline_ed1_1_final
BASELINE_ED2_1 = baseline_ed2_1_final

print("\n--- Constantes de Baseline Definidas para o Ano:", ano, "---")
if arquivo_ano_anterior_encontrado:
     print(f"   (Valores baseados nos totais calculados para {ano_anterior})")
else:
     print(f"   (Valores baseados no histórico {BASELINE_HISTORICO_ED1_1} / {BASELINE_HISTORICO_ED2_1})")
print(f"   - Baseline ED1.1 (Emissões): {BASELINE_ED1_1:,.2f} KgCO2eq")
print(f"   - Baseline ED2.1 (Custo): R$ {BASELINE_ED2_1:,.2f}")


--- Definindo Constantes de Baseline ---
   Arquivo de relatório para 2022 não encontrado (padrão: dadosViagens/dados_viagens2022\relatorio_metricas_scores_UFCG_2022_*.csv). Usando baseline histórico fixo.

--- Constantes de Baseline Definidas para o Ano: 2023 ---
   (Valores baseados no histórico 285440.323 / 1738431.16)
   - Baseline ED1.1 (Emissões): 285,440.32 KgCO2eq
   - Baseline ED2.1 (Custo): R$ 1,738,431.16


## 6. Cálculo dos Scores de Desempenho

In [72]:
scores = {}
variacoes = {}

# Usa o metrics_df criado na Célula 9
if not df.empty and not metrics_df.empty:
    print("\n--- Calculando Scores ---")
    
    # --- Helper para buscar valor bruto da métrica --- 
    def get_raw_metric(metric_code_prefix):
        try:
            # Encontra a linha que começa com o código (ex: 'ED1.1')
            metric_row = metrics_df[metrics_df['Indicador'].str.startswith(metric_code_prefix)]
            if not metric_row.empty: return metric_row['Valor Bruto'].iloc[0]
            else: print(f"   ⚠️ Métrica '{metric_code_prefix}' não encontrada."); return 0
        except Exception as e: print(f"   ❌ Erro ao buscar métrica '{metric_code_prefix}': {e}"); return 0

    # --- Score DF1.5 --- 
    total_viagens = get_raw_metric('ED2.3_Total_Trips')
    urgentes_sem_justificativa_count = get_raw_metric('DF1.5_Urgent_Trips_wo_Justif_Count')
    if total_viagens > 0:
        df1_5_proporcao = urgentes_sem_justificativa_count / total_viagens
        df1_5_score = max(0, 1.0 - df1_5_proporcao) 
    else: df1_5_proporcao = 0.0; df1_5_score = 1.0 
    variacoes['DF1.5_Proporção_Urg_s_Just'] = df1_5_proporcao
    scores['DF1.5_Score'] = df1_5_score
    print(f"   - Score DF1.5: {df1_5_score:.4f} (Proporção: {df1_5_proporcao:.2%})")

    # --- Score ED1.1 --- 
    metrica_atual_emissao = get_raw_metric('ED1.1_Total_Emissions')
    score_ed1_1 = 0.0; variacao_ed1_1 = np.nan
    if BASELINE_ED1_1 > 0: 
        variacao_ed1_1 = (metrica_atual_emissao - BASELINE_ED1_1) / BASELINE_ED1_1
        if metrica_atual_emissao <= BASELINE_ED1_1: score_ed1_1 = 1.0
        else: score_ed1_1 = max(0, 1.0 - (variacao_ed1_1 / 2.0)) # Ajuste '/ 2.0' se necessário
    elif metrica_atual_emissao == 0: score_ed1_1 = 1.0; variacao_ed1_1 = 0.0
    variacoes['ED1.1_Variação_vs_Baseline'] = variacao_ed1_1
    scores['ED1.1_Score'] = score_ed1_1
    print(f"   - Score ED1.1: {score_ed1_1:.4f} (Variação: {variacao_ed1_1:+.2%})")

    # --- Score ED2.1 --- 
    metrica_atual_custo = get_raw_metric('ED2.1_Total_Costs')
    score_ed2_1 = 0.0; variacao_ed2_1 = np.nan
    if BASELINE_ED2_1 > 0:
        variacao_ed2_1 = (metrica_atual_custo - BASELINE_ED2_1) / BASELINE_ED2_1
        if metrica_atual_custo <= BASELINE_ED2_1: score_ed2_1 = 1.0
        else: score_ed2_1 = max(0, 1.0 - (variacao_ed2_1 / 2.0)) # Ajuste '/ 2.0' se necessário
    elif metrica_atual_custo == 0: score_ed2_1 = 1.0; variacao_ed2_1 = 0.0
    variacoes['ED2.1_Variação_vs_Baseline'] = variacao_ed2_1
    scores['ED2.1_Score'] = score_ed2_1
    print(f"   - Score ED2.1: {score_ed2_1:.4f} (Variação: {variacao_ed2_1:+.2%})")
    
    print("✅ Scores calculados.")

    # --- Cria DataFrames Finais --- 
    scores_df = pd.DataFrame(scores.items(), columns=['Indicador', 'Score'])
    variacoes_df = pd.DataFrame(variacoes.items(), columns=['Indicador', 'Variação'])
    baselines_df = pd.DataFrame([
        {'Indicador': 'ED1.1_Baseline_KgCO2eq', 'Valor': BASELINE_ED1_1},
        {'Indicador': 'ED2.1_Baseline_R$', 'Valor': BASELINE_ED2_1}
    ])
    
    print("\n--- Tabela de Scores --- ")
    print(scores_df.to_string(index=False))

else:
    print("⚠️ Cálculo de scores pulado.")
    scores_df = pd.DataFrame(columns=['Indicador', 'Score'])
    variacoes_df = pd.DataFrame(columns=['Indicador', 'Variação'])
    baselines_df = pd.DataFrame(columns=['Indicador', 'Valor'])


--- Calculando Scores ---
   - Score DF1.5: 1.0000 (Proporção: 0.00%)
   - Score ED1.1: 1.0000 (Variação: -5.19%)
   - Score ED2.1: 1.0000 (Variação: -10.78%)
✅ Scores calculados.

--- Tabela de Scores --- 
  Indicador  Score
DF1.5_Score    1.0
ED1.1_Score    1.0
ED2.1_Score    1.0


## 7. Combinar Relatórios e Salvar

In [73]:
print("\n--- Combinando e Salvando Relatório Final ---")

# Prepara DataFrames para concatenação (usa os DFs finais da seção anterior)
metrics_df_final = metrics_df[['Indicador', 'Valor Formatado']].rename(columns={'Indicador': 'Indicador/Métrica', 'Valor Formatado': 'Valor'})
metrics_df_final['Tipo'] = 'Métrica Bruta'

variacoes_df_final = variacoes_df.rename(columns={'Indicador': 'Indicador/Métrica', 'Variação': 'Valor'})
variacoes_df_final['Tipo'] = 'Variação vs Baseline'
# Formata variação como % (trata NaNs)
variacoes_df_final['Valor'] = variacoes_df_final['Valor'].apply(lambda x: f"{x:.2%}" if pd.notna(x) else "N/A")

scores_df_final = scores_df.rename(columns={'Indicador': 'Indicador/Métrica', 'Score': 'Valor'})
scores_df_final['Tipo'] = 'Score (0 a 1)'
scores_df_final['Valor'] = scores_df_final['Valor'].round(4) # Arredonda score

baselines_df_final = baselines_df.rename(columns={'Indicador': 'Indicador/Métrica', 'Valor': 'Valor'})
baselines_df_final['Tipo'] = 'Baseline'
# Formata baseline
baselines_df_final['Valor'] = baselines_df_final.apply(
    lambda row: f"R$ {row['Valor']:,.2f}" if 'R$' in row['Indicador/Métrica'] else (f"{row['Valor']:,.2f}" if pd.notna(row['Valor']) else "N/A"), axis=1
)


# Junta tudo
relatorio_final = pd.concat([
    metrics_df_final,
    baselines_df_final,
    variacoes_df_final,
    scores_df_final
], ignore_index=True)

# Reordena colunas e Indicadores para melhor visualização
ordem_indicadores = [
    # Dimensão 1
    'ED1.1_Total_Emissions_KgCO2eq', 'ED1.1_Baseline_KgCO2eq', 'ED1.1_Variação_vs_Baseline', 'ED1.1_Score',
    'ED1.2_Avoidable_Emissions_KgCO2eq',
    'ED1.3_Avg_Emissions_per_Traveler',
    'ED1.4_Avg_Emissions_per_Trip',
    # Dimensão 2
    'ED2.1_Total_Costs_R$', 'ED2.1_Baseline_R$', 'ED2.1_Variação_vs_Baseline', 'ED2.1_Score',
    'ED2.2_Avg_Cost_per_Traveler',
    'ED2.3_Total_Trips',
    'ED2.4_Avg_Cost_per_Trip',
    'ED2.5_Avg_Cost_per_Km',
    # Dimensão 3
    'ED3.1_Total_Distance_Km',
    'ED3.2_Avg_Distance_per_Traveler',
    'ED3.3_Avg_Distance_per_Trip',
    # Dimensão 4
    'DF1.4_Urgent_Trips_Percent', 'DF1.4_Urgent_Trips_Count',
    'DF1.5_Urgent_Trips_wo_Justif_Percent', 'DF1.5_Urgent_Trips_wo_Justif_Count', 'DF1.5_Proporção_Urg_s_Just', 'DF1.5_Score'
]
# Converte para Categorical para ordenar corretamente, mantendo os não listados no final
relatorio_final['Indicador/Métrica'] = pd.Categorical(relatorio_final['Indicador/Métrica'], categories=ordem_indicadores, ordered=True)
relatorio_final.sort_values('Indicador/Métrica', inplace=True)

relatorio_final = relatorio_final[['Indicador/Métrica', 'Tipo', 'Valor']] # Reordena colunas

print("\n--- Relatório Final Combinado --- ")
# Ajusta opções para ver tudo
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None, 'display.width', 1000):
    print(relatorio_final.to_string(index=False, na_rep='N/A')) # Mostra NaN como N/A

# Salvar o relatório
try:
    TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
    nome_arquivo_saida = f"relatorio_metricas_scores_{orgao}_{ano}_{TIMESTAMP}.csv"
    caminho_saida = os.path.join(pasta_dados, nome_arquivo_saida) 
    
    # Salva com separador ';' e decimal ',' para Excel em português
    relatorio_final.to_csv(caminho_saida, index=False, sep=';', decimal=',') 
    print(f"\n✅ Relatório final salvo com sucesso em: '{caminho_saida}'")
except Exception as e:
    print(f"❌ Erro ao salvar o relatório final: {e}")



--- Combinando e Salvando Relatório Final ---

--- Relatório Final Combinado --- 
                   Indicador/Métrica                 Tipo              Valor
       ED1.1_Total_Emissions_KgCO2eq        Métrica Bruta 270,626.40 KgCO2eq
              ED1.1_Baseline_KgCO2eq             Baseline         285,440.32
          ED1.1_Variação_vs_Baseline Variação vs Baseline             -5.19%
                         ED1.1_Score        Score (0 a 1)                1.0
   ED1.2_Avoidable_Emissions_KgCO2eq        Métrica Bruta     224.41 KgCO2eq
    ED1.3_Avg_Emissions_per_Traveler        Métrica Bruta   1,252.90 KgCO2eq
        ED1.4_Avg_Emissions_per_Trip        Métrica Bruta     845.71 KgCO2eq
                ED2.1_Total_Costs_R$        Métrica Bruta    R$ 1,551,094.54
                   ED2.1_Baseline_R$             Baseline    R$ 1,738,431.16
          ED2.1_Variação_vs_Baseline Variação vs Baseline            -10.78%
                         ED2.1_Score        Score (0 a 1)             